In [1]:
import json
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, UMT5Config, UMT5ForConditionalGeneration, Seq2SeqTrainingArguments,Seq2SeqTrainer, EarlyStoppingCallback 
import torch
from datasets import Dataset, DatasetDict
import transformers, dataclasses

D:\Projects\kaz-gec\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Dataset

In [2]:
def load_jsonl(path: str) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        records = [json.loads(line) for line in f]
    return pd.DataFrame(records)

train_df = load_jsonl("../data/processed/train.jsonl")
val_df = load_jsonl("../data/processed/val.jsonl")
test_real_df = load_jsonl("../data/processed/test_real.jsonl")
test_regression_df = load_jsonl("../data/processed/test_regression.jsonl")

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test_real": Dataset.from_pandas(test_real_df),
    "test_regression": Dataset.from_pandas(test_regression_df),
})

In [3]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 168326 entries, 0 to 168325
Data columns (total 3 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   corrupted  168326 non-null  str  
 1   clean      168326 non-null  str  
 2   source     168326 non-null  str  
dtypes: str(3)
memory usage: 85.1 MB


---
## Tokenizer and model

In [ ]:
MODEL_NAME = "google/umt5-small"

In [5]:
MAX_LENGTH = 128 

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(examples):
    model_inputs = tokenizer(
        examples["corrupted"],
        max_length=MAX_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=examples["clean"],
        max_length=MAX_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [6]:
config = UMT5Config.from_pretrained(MODEL_NAME)
model = UMT5ForConditionalGeneration.from_pretrained(MODEL_NAME, config=config)

Loading weights: 100%|██████████| 306/306 [00:00<00:00, 50997.62it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [7]:
def unify_columns(ds):
    rename_map = {}
    if "input" in ds.column_names:
        rename_map["input"] = "corrupted"
    if "target" in ds.column_names:
        rename_map["target"] = "clean"
    return ds.rename_columns(rename_map) if rename_map else ds

dataset = DatasetDict({split: unify_columns(ds) for split, ds in dataset.items()})

In [8]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
)

In [9]:
tokenized_datasets = DatasetDict({
    split: ds.map(preprocess, batched=True, remove_columns=ds.column_names)
    for split, ds in dataset.items()
})

print(tokenized_datasets)
print(tokenized_datasets["train"][0])

Map: 100%|██████████| 105/105 [00:00<00:00, 13133.39 examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 168326
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2400
    })
    test_real: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 120
    })
    test_regression: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 105
    })
})
{'input_ids': [297, 110099, 20944, 273, 284, 280, 1174, 90123, 1109, 105042, 463, 16970, 463, 1174, 292, 914, 296, 150299, 274, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [297, 110099, 20944, 273, 284, 280, 1174, 90123, 1109, 76954, 10056, 54745, 73180, 274, 1]}


---

## Training

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="../models/umt5-gec-small",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    bf16=True,
    optim="adafactor",                 
    num_train_epochs=1,
    learning_rate=3e-4,                
    warmup_steps=500,
    lr_scheduler_type="linear",
    eval_strategy="steps", eval_steps=1000,
    save_strategy="steps", save_steps=1000,
    save_total_limit=2, logging_steps=50,
    predict_with_generate=True,        # <-- критично
    generation_max_length=MAX_LENGTH,
    generation_num_beams=1,            
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
)

In [ ]:
model.config.use_cache = False

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

In [ ]:
import torch
from tqdm.auto import tqdm

device = model.device
model.eval()

def generate_corrections(texts, batch_size=16, max_new_tokens=MAX_LENGTH):
    outputs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(
            batch,
            max_length=MAX_LENGTH,
            truncation=True,
            padding=True,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                num_beams=4,
            )

        decoded = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        outputs.extend(decoded)
    return outputs

In [ ]:
test_real_sources = test_real_df["corrupted"].tolist()
test_real_targets = test_real_df["clean"].tolist()

predictions = generate_corrections(test_real_sources)

test_real_results = test_real_df.copy()
test_real_results["prediction"] = predictions

# Глазами — первые 10 примеров
for i in range(10):
    print("SOURCE: ", test_real_results.iloc[i]["corrupted"])
    print("TARGET: ", test_real_results.iloc[i]["clean"])
    print("PRED:   ", test_real_results.iloc[i]["prediction"])
    print("-" * 60)

In [ ]:
test_regression_sources = test_regression_df["input"].tolist()  # исходное имя колонки до unify
test_regression_targets = test_regression_df["target"].tolist()

reg_predictions = generate_corrections(test_regression_sources)

test_regression_results = test_regression_df.copy()
test_regression_results["prediction"] = reg_predictions

# Ключевая проверка: сколько раз модель ИЗМЕНИЛА уже чистый текст (регрессия)
test_regression_results["unchanged"] = (
    test_regression_results["prediction"].str.strip()
    == test_regression_results["input"].str.strip()
)

print("Доля неизменённых (не тронула чистый текст):", test_regression_results["unchanged"].mean())

# Разбивка по type (clean vs general из README)
print(test_regression_results.groupby("type")["unchanged"].mean())

In [ ]:
model.config.use_cache = True
model.eval()

def generate_corrections(texts, batch_size=16, max_new_tokens=MAX_LENGTH):
    outputs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(
            batch, max_length=MAX_LENGTH, truncation=True,
            padding=True, return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                num_beams=4,
                repetition_penalty=1.3,
                no_repeat_ngram_size=3,
            )
        decoded = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        outputs.extend(decoded)
    return outputs

In [ ]:
model.config.use_cache = True
test_preds = generate_corrections(test_real_sources[:5])
for s, t, p in zip(test_real_sources[:5], test_real_targets[:5], test_preds):
    print("SRC:", s)
    print("TGT:", t)
    print("PRED:", p)
    print("-"*40)

In [ ]:
model.eval()
model.config.use_cache = True

text = test_real_sources[0]
inputs = tokenizer(text, return_tensors="pt", max_length=MAX_LENGTH, truncation=True).to(device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=64, num_beams=1, do_sample=False)

print(tokenizer.decode(out[0], skip_special_tokens=True))
print(tokenizer.decode(out[0], skip_special_tokens=False))  # со спец-токенами — важно увидеть

In [ ]:
text_train = train_df["corrupted"].iloc[0]
inputs = tokenizer(text_train, return_tensors="pt", max_length=MAX_LENGTH, truncation=True).to(device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=64, num_beams=1, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))

In [ ]:
print(trainer.state.best_model_checkpoint)
print(trainer.state.best_metric)
print(trainer.state.global_step)

In [ ]:
metrics = trainer.evaluate()
print(metrics)

In [ ]:
model.eval()
model.config.use_cache = True

text = test_real_sources[0]
inputs = tokenizer(text, return_tensors="pt", max_length=MAX_LENGTH, truncation=True).to(device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=64, num_beams=1, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))

In [ ]:
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=64,
        num_beams=1,
        do_sample=False,
        repetition_penalty=1.5,      # сильнее, чем пробовали (было 1.3)
        no_repeat_ngram_size=2,      # жёстче (было 3)
    )
print(tokenizer.decode(out[0], skip_special_tokens=True))

In [ ]:
import transformers
print(transformers.__version__)   # должно быть 4.57.6, НЕ 5.x

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

CKPT = "../models/umt5-gec-small/best"
tok = AutoTokenizer.from_pretrained(CKPT)
model = AutoModelForSeq2SeqLM.from_pretrained(CKPT).to("cuda").eval()
model.config.use_cache = True

@torch.no_grad()
def correct(text):
    x = tok(text, return_tensors="pt", truncation=True, max_length=128).to(model.device)
    y = model.generate(x.input_ids, attention_mask=x.attention_mask,
                       max_new_tokens=128, num_beams=4,
                       no_repeat_ngram_size=3, early_stopping=True)
    return tok.decode(y[0], skip_special_tokens=True)

# проверка
for t in ["Шамалыдан сон біз дүнйеге келдик.",
          "«Қазақ» газетінін редакторы болды."]:
    print("IN :", t)
    print("OUT:", correct(t))
    print("-"*40)